# LLM Observability Lab — Beginner's Sequential Setup Guide

This notebook is a **step-by-step guide** for someone who has never run this lab before. Follow the steps in order. Every command, every URL, every screen you need to click is listed here.

**You need:**
- Docker Desktop installed and running
- Python 3.11+ installed
- A Groq account (free at https://console.groq.com)
- A LangSmith account (free at https://smith.langchain.com)
- This repo cloned to `d:\Observation_Layer`

---

## The 10-Step Sequence

| Step | What you do | Where | Time |
|---|---|---|---|
| 1 | Get Groq API key | console.groq.com (cloud) | 2 min |
| 2 | Get LangSmith API key | smith.langchain.com (cloud) | 2 min |
| 3 | Start Langfuse locally | terminal (Langfuse directory) | 5 min |
| 4 | Create Langfuse account + project + API keys | browser (localhost:3000) | 5 min |
| 5 | Paste all keys into `.env`, then run `sync-keys.ps1` | terminal (this repo) | 2 min |
| 6 | Start the observability stack | terminal (this repo) | 2 min |
| 7 | Open all the web UIs and verify | browser (5 URLs) | 3 min |
| 8 | Run the example scripts | terminal (this repo) | 5 min |
| 9 | Check dashboards and traces | browser | 5 min |
| 10 | Clean wipe for a fresh start | terminal | 2 min |

---
## Step 1 — Get Your Groq API Key (Cloud)

1. Open **https://console.groq.com** in your browser
2. Sign in (Google account or email)
3. On the left sidebar, click **API Keys**
4. Click **Create API Key**
5. Give it a name (e.g., `observability-lab`)
6. **Copy the key** — it looks like `gsk_abc123...` — you won't see it again
7. Save it somewhere safe temporarily (notepad is fine)

**You now have:**
```
GROQ_API_KEY = gsk_your_key_here
```

> The default model `allam-2-7b` is free and has a generous rate limit. No credit card needed.

---
## Step 2 — Get Your LangSmith API Key (Cloud)

1. Open **https://smith.langchain.com** in your browser
2. Sign in (Google or GitHub account)
3. Click your profile (top right) → **Settings**
4. On the left, click **API Keys**
5. Click **Create API Key**
6. Name it `observability-lab`
7. **Copy the key** — it looks like `lsv2_pt_xxx...`
8. Also note your **project name** — if you don't have one, create a new project called `llm-observability-lab`

**You now have:**
```
LANGSMITH_API_KEY = lsv2_pt_your_key_here
LANGSMITH_PROJECT = llm-observability-lab
```

---
## Step 3 — Start Langfuse Locally

Langfuse is self-hosted. It needs to be running BEFORE you start this lab's Docker stack, because this lab's collector connects to it.

### 3a — Clone Langfuse (if not already done)

Open a terminal and run:
```powershell
cd d:\
git clone https://github.com/langfuse/langfuse.git
cd langfuse
```

> If you already have Langfuse cloned and running, skip to Step 4.

### 3b — Start Langfuse

From the Langfuse directory:
```powershell
cd d:\langfuse
docker compose up -d
```

**Expected output:**
```
[+] Running 8/8
 ✔ Container langfuse-postgres-1       Healthy
 ✔ Container langfuse-clickhouse-1     Healthy
 ✔ Container langfuse-redis-1          Started
 ✔ Container langfuse-minio-1          Started
 ✔ Container langfuse-langfuse-web-1   Started
 ✔ Container langfuse-langfuse-worker-1 Started
```

### 3c — Verify Langfuse is up

Open **http://localhost:3000** in your browser.

**You should see:** the Langfuse login/signup page.

> If port 3000 is already in use by something else, Langfuse will fail to start. Check `docker ps` and stop whatever is using port 3000.

---
## Step 4 — Create Langfuse Account, Project, and API Keys

Now that Langfuse is running at http://localhost:3000, you need to create an account and get API keys.

### 4a — Sign up

1. Open **http://localhost:3000**
2. Click **Sign up**
3. Fill in:
   - Name: any name (e.g. `Observability Lab`)
   - Email: any email (e.g. `you@example.com`)
   - Password: choose any strong password and write it down
4. Click **Sign up**

# ---- Pricing (USD per 1M tokens; premium-model-level for visible dashboards) ----
PRICE_INPUT_PER_M=5.00
PRICE_OUTPUT_PER_M=15.00

# ---- Safety ----
ALLOW_LIVE_CALLS=1

---
## Step 6 — Start the Observability Stack

This starts Phoenix, Prometheus, Grafana, and the OTel Collector — all in this repo's Docker Compose.

Open a terminal:
```powershell
cd d:\Observation_Layer
docker compose up -d
```

**Expected output:**
```
[+] Running 4/4
 ✔ Container observation_layer-phoenix-1        Started
 ✔ Container observation_layer-prometheus-1     Started
 ✔ Container observation_layer-grafana-1        Started
 ✔ Container observation_layer-otel-collector-1 Started
```

### Verify the collector is healthy

```powershell
docker logs observation_layer-otel-collector-1 --tail 3
```

**You should see:**
```
Starting HTTP server    endpoint: [::]:4318
Everything is ready. Begin running and processing data.
```

**If you see `address already in use`:**
```powershell
docker stop observation_layer-otel-collector-1
docker rm observation_layer-otel-collector-1
docker compose up -d otel-collector
```

**If port 9090 or 3000 is already in use** (by Langfuse's minio or another service):
- Edit `docker-compose.yml` and change the Prometheus port mapping (e.g., `"9095:9090"`)
- Grafana is already on 3001 to avoid conflict with Langfuse on 3000

---
## Step 7 — Open All the Web UIs and Verify

Open each of these in your browser. Here's what you should see at each URL:

### 7a — Phoenix (trace waterfall)

| | |
|---|---|
| URL | http://localhost:6006 |
| Login | None needed |
| What you see | "Projects" page with a `default` project, 0 traces |

### 7b — Langfuse (traces + dashboards)

| | |
|---|---|
| URL | http://localhost:3000 |
| Login | Email + password you created in Step 4a |
| What you see | Your `llm-obs-lab` project, Tracing page (empty) |

### 7c — Prometheus (raw metrics)

| | |
|---|---|
| URL | http://localhost:9095/targets |
| Login | None needed |
| What you see | A table with `otel-collector` job, State = **UP** in green |

> If the target shows DOWN, the collector didn't start properly. Check `docker logs observation_layer-otel-collector-1`.

### 7d — Grafana (SLO dashboards)

| | |
|---|---|
| URL | http://localhost:3001 |
| Login | Username: `admin`, Password: `admin` |
| What you see | Home page → click **Dashboards** on the left |

After login:
1. Click **Dashboards** in the left sidebar
2. You should see a folder called **LLM Observability**
3. Expand it → 3 dashboards:
   - LLM Service Overview
   - Token and Cost Analytics
   - Failure and SLO Diagnostics
4. All panels will show **"No data"** or **0** — that's normal, no scripts have run yet

### 7e — LangSmith (SaaS traces)

| | |
|---|---|
| URL | https://smith.langchain.com |
| Login | Your LangSmith account |
| What you see | Your `llm-observability-lab` project, 0 traces |

**All 5 UIs verified? Move to Step 8.**

### Script 6 — 500 synthetic observations (no API calls — safe)
```powershell
python examples\06_synthetic_metrics.py
```
**Expected:** 500 synthetic observations, ~469 success / ~31 errors, SLO report printed.

### Script 7 — Evaluation with scores (needs live calls)
```powershell
python examples\07_evaluation.py
```
**Expected:** 5 test cases, each with 4 heuristic checks (success, completeness, relevance, conciseness).
Scores are logged to Langfuse as trace attributes and appear in the Scores column.

### Run the test suite
```powershell
python -m pytest -v
```
**Expected:** `36 passed` (or similar).

---
## Step 9 — Check Dashboards and Traces

After running the scripts, wait 5-10 seconds for data to flush, then check each UI:

### 9a — Phoenix (http://localhost:6006)

1. Go to Projects → `default` → **Traces** tab
2. You should see multiple traces: `single_call.workflow`, `multi_call.workflow`, `failure_workflow`, `batch_sequential.workflow`, `batch_concurrent.workflow`
3. Click `batch_concurrent.workflow` → see the waterfall:
   - Parent span at top
   - 15 child `Completions` spans below (one per Groq call)
   - Each child shows: input JSON, output JSON, model, token counts, latency
4. Click a `failure_workflow` trace → see error spans in red with exception details

### 9b — Langfuse (http://localhost:3000)

1. Go to your project → **Tracing**
2. You should see traces with timestamps, names, observation counts, latency, tokens
3. Click **Dashboards** on the left → 3 built-in dashboards:
   - **Cost Dashboard** — shows trace/observation counts, cost by model (shows $0 unless you configure model pricing in Langfuse settings)
   - **Latency Dashboard** — P95 latency by use case, by model, output tokens/s
   - **Usage Dashboard** — total trace count, observation count, trends over time

### 9c — Grafana (http://localhost:3001)

1. Go to Dashboards → **LLM Service Overview**
   - **Total Requests**: ~500+ (from all scripts + synthetic)
   - **Success Rate**: ~93% (yellow — below 99.5% SLO because synthetic data has ~6% errors)
   - **P95 Latency**: ~0.9s (green — under 4s SLO)
   - **Requests by Workflow**: bar chart showing each workflow
   - **Latency P50/P95/P99**: time-series chart

2. Go to **Token and Cost Analytics**
   - **Total Tokens**: ~185,000
   - **Input Tokens**: ~26,000
   - **Output Tokens**: ~159,000
   - **Estimated Cost**: ~$0.034

3. Go to **Failure and SLO Diagnostics**
   - **Error Count**: ~33
   - **Retry Attempts**: ~116
   - **SLO: Success Rate**: ~93% (yellow/red — violated)
   - **SLO: % Under 4s**: ~100% (green — met)
   - **Error Budget Consumed**: >100% (red — exhausted)
   - **Total Samples**: ~500+ (green — above 100 minimum)
   - **Errors by Type**: chart showing timeout, rate_limit, transient_error

### 9d — Prometheus (http://localhost:9095)

1. Go to the **Query** page
2. Try these queries:
   - `sum(llm_obs_llm_requests_total)` → total request count
   - `sum by (result) (llm_obs_llm_requests_total)` → breakdown by success/error
   - `sum(llm_obs_llm_tokens_total)` → total tokens

### 9e — LangSmith (https://smith.langchain.com)

1. Go to your `llm-observability-lab` project
2. You should see traces with runs, model, tokens, latency

**All dashboards showing data? You're done!** 🎉

### 10d — Fresh restart

```powershell
# 1. Start Langfuse fresh
cd d:\langfuse
docker compose up -d

# 2. Open http://localhost:3000, sign up again, create org + project + API keys

# 3. Update .env with the new Langfuse keys, then sync to collector
cd d:\Observation_Layer
# -> edit .env with new LANGFUSE_PUBLIC_KEY and LANGFUSE_SECRET_KEY
.\infrastructure\sync-keys.ps1

# 4. Start the observability stack
docker compose up -d

# 5. Run scripts again
$env:PYTHONPATH = "d:\Observation_Layer"
$env:PYTHONIOENCODING = "utf-8"
python examples\06_synthetic_metrics.py   # quick data populate
```

---
## Quick Reference — All Commands in One Place

### One-time setup
```powershell
# 1. Get Groq key from console.groq.com
# 2. Get LangSmith key from smith.langchain.com

# 3. Start Langfuse
cd d:\langfuse
docker compose up -d
# -> open http://localhost:3000, sign up, create org + project + API keys

# 4. Configure this repo
cd d:\Observation_Layer
cp .env.example .env
# -> edit .env with all 4 keys (GROQ, LANGFUSE_PUBLIC, LANGFUSE_SECRET, LANGSMITH)
.\infrastructure\sync-keys.ps1
# -> auto-generates collector.env from .env + restarts collector

# 5. Create venv (first time only)
python -m venv .venv
& .venv\Scripts\Activate.ps1
pip install -e ".[dev]"
```

### Daily start (every time you reboot)
```powershell
# Start Langfuse (if not already running)
cd d:\langfuse
docker compose up -d

# Start observability stack
cd d:\Observation_Layer
docker compose up -d

# Activate venv + set env vars
& d:\Observation_Layer\.venv\Scripts\Activate.ps1
$env:PYTHONPATH = "d:\Observation_Layer"
$env:PYTHONIOENCODING = "utf-8"
```

### Run scripts — what each one does

| # | Command | What it does | Live API? | What you'll see |
|---|---|---|---|---|
| 01 | `python examples\01_single_call.py` | Makes **1 Groq call** with a simple prompt. Verifies all 3 trace platforms (Langfuse, Phoenix, LangSmith) receive the trace. | Yes | 1 trace with parent + child span, token counts, cost |
| 02 | `python examples\02_multiple_calls.py` | Makes **3 sequential calls** under one workflow span. Shows parent/child trace structure and how tokens are summed without double counting. | Yes | 1 trace with 3 child spans, workflow summary with total tokens + cost |
| 03 | `python examples\03_concurrent_calls.py` | Runs **5 sequential vs 5 concurrent** calls. Compares p50/p95 latency, throughput, and speedup. Each concurrent call gets its own child span. | Yes | Two sections (seq vs conc) with latency percentiles, throughput, speedup ratio |
| 04 | `python examples\04_failures.py` | Simulates **timeout, 429 rate-limit, and transient error** scenarios using a fake provider. Each retry attempt is a distinct child span. Never calls the real API. | **No** (fake provider) | 3 failure traces with retry spans, error status, exception details |
| 05 | `python examples\05_batch_dashboard.py` | Runs **30 calls** (15 sequential + 15 concurrent) with longer prompts and higher max_tokens. Generates enough volume for meaningful dashboard data. | Yes | Per-call breakdown table, grand total tokens + cost, speedup ratio |
| 06 | `python examples\06_synthetic_metrics.py` | Generates **500 synthetic observations** with realistic latency/token distributions (~95% success, ~5% errors). Populates Grafana dashboards without making any API calls. | **No** (offline) | 500 synthetic metrics, SLO report printed |
| 07 | `python examples\07_evaluation.py` | Runs **5 test cases** with known prompts, checks each response with 4 heuristic evaluators (success, completeness, relevance, conciseness). Scores logged to Langfuse. | Yes | 5 eval cases, each with 4 scores, summary table with pass/fail |
| — | `python -m pytest -v` | Runs the **test suite** (36 tests covering usage extraction, pricing, metadata, instrumentation, metrics, SLOs, failures). | **No** | `36 passed` |

### Recommended run order

**First time / testing the pipeline:**
```powershell
python examples\01_single_call.py       # verify traces reach all 3 platforms
python examples\04_failures.py          # see error traces (no API cost)
python -m pytest -v                     # verify everything works
```

**Populating dashboards (no API cost):**
```powershell
python examples\06_synthetic_metrics.py # 500 synthetic observations for Grafana
```

**Full demonstration (uses live API quota):**
```powershell
python examples\01_single_call.py       # single call
python examples\02_multiple_calls.py    # 3 sequential
python examples\03_concurrent_calls.py  # 5 concurrent vs sequential
python examples\05_batch_dashboard.py   # 30 calls for dashboard volume
python examples\07_evaluation.py        # 5 eval cases with scores
```

**Quick commands (copy-paste):**
```powershell
python examples\01_single_call.py          # 1 Groq call
python examples\02_multiple_calls.py       # 3 sequential calls
python examples\03_concurrent_calls.py     # 5 concurrent vs sequential
python examples\04_failures.py             # failures + retries (no API)
python examples\05_batch_dashboard.py      # 30 calls for dashboards
python examples\06_synthetic_metrics.py    # 500 synthetic (no API)
python examples\07_evaluation.py           # 5 eval cases with scores
python -m pytest -v                         # run tests
```

### 9a — Phoenix (http://localhost:6006)

1. Go to Projects -> `default` -> **Traces** tab
2. You should see multiple traces: `single_call.workflow`, `multi_call.workflow`, `failure_workflow`, `batch_sequential.workflow`, `batch_concurrent.workflow`, `evaluation.workflow`
3. Click `batch_concurrent.workflow` -> see the waterfall:
   - Parent span at top
   - 15 child `Completions` spans below (one per Groq call)
   - Each child shows: input JSON, output JSON, model, token counts, latency
4. Click a `failure_workflow` trace -> see error spans in red with exception details
5. Click the `evaluation.workflow` trace -> see 5 `eval.*` case spans, each with a child `Completions` span and score attributes (eval.success, eval.relevance, eval.completeness, eval.conciseness)

### 9b — Langfuse (http://localhost:3000)

1. Go to your project -> **Tracing**
2. You should see traces with timestamps, names, observation counts, latency, tokens
3. Look for the `evaluation.workflow` trace -> click it -> check the Scores column for eval scores
4. Click **Dashboards** on the left -> 3 built-in dashboards:

---
## Step 1 — Configure Your `.env` File

The `.env` file holds all credentials and settings. Here's what each key does and where to get it.

**Copy the template:**
```powershell
cp .env.example .env
```

**Then edit `.env` and fill in these values:**

| Key | What it is | Where to get it | Example |
|---|---|---|---|
| `GROQ_API_KEY` | Your Groq API key | https://console.groq.com → API Keys | `gsk_abc123...` |
| `GROQ_MODEL` | Which model to call | Groq docs | `allam-2-7b` |
| `OTEL_EXPORTER_OTLP_ENDPOINT` | Where the collector listens | Keep default | `http://localhost:4318` |
| `OTEL_SERVICE_NAME` | Name shown in all platforms | Keep default | `llm-observability-lab` |
| `LANGFUSE_PUBLIC_KEY` | Langfuse project public key | Langfuse UI → Settings → API Keys | `pk-lf-...` |
| `LANGFUSE_SECRET_KEY` | Langfuse project secret key | Same page, shown once at creation | `sk-lf-...` |
| `LANGFUSE_BASE_URL` | Langfuse URL | Keep default (self-hosted) | `http://localhost:3000` |
| `LANGSMITH_API_KEY` | LangSmith API key | https://smith.langchain.com → Settings | `lsv2_pt_...` |
| `LANGSMITH_PROJECT` | LangSmith project name | Create in LangSmith UI | `llm-observability-lab` |
| `PRICE_INPUT_PER_M` | Cost per 1M input tokens (USD) | Groq pricing page | `5.00` |
| `PRICE_OUTPUT_PER_M` | Cost per 1M output tokens (USD) | Groq pricing page | `15.00` |
| `ALLOW_LIVE_CALLS` | `1` = real Groq calls, `0` = dry-run | Set to `1` for this walkthrough | `1` |

> **Important:** Never commit `.env` to git. It's already in `.gitignore`.

---
## Step 2 — Generate the Langfuse Auth Header

The OTel Collector needs a **Base64-encoded** auth header to send traces to Langfuse. It's built from `public_key:secret_key`.

**Run this in a terminal to generate it:**

```powershell
$pub = "pk-lf-YOUR_PUBLIC_KEY"
$sec = "sk-lf-YOUR_SECRET_KEY"
$raw = "$($pub):$($sec)"
[Convert]::ToBase64String([Text.Encoding]::UTF8.GetBytes($raw))
```

**Expected output:**
```
cGstbGYt...base64string...==
```

**Then put this value in `infrastructure/collector.env`:**
```
LANGFUSE_HOST=http://langfuse-web:3000
LANGFUSE_AUTH_HEADER=cGstbGYt...your_base64...==
PHOENIX_ENDPOINT=phoenix:4317
LANGSMITH_OTLP_ENDPOINT=https://api.smith.langchain.com/otel
LANGSMITH_API_KEY=lsv2_pt_YOUR_KEY
LANGSMITH_PROJECT=llm-observability-lab
```

> `collector.env` is also gitignored — it contains real secrets.

---
## Step 3 — Start the Docker Stack

This starts Phoenix, Prometheus, Grafana, and the OTel Collector.

**Run in a terminal:**
```powershell
cd d:\Observation_Layer
docker compose up -d
```

**Expected output:**
```
[+] up 4/4
 ✔ Container observation_layer-phoenix-1        Running
 ✔ Container observation_layer-prometheus-1     Started
 ✔ Container observation_layer-otel-collector-1 Started
 ✔ Container observation_layer-grafana-1        Started
```

**Verify the collector is healthy:**
```powershell
docker logs observation_layer-otel-collector-1 --tail 3
```

**Expected:**
```
Starting HTTP server    endpoint: [::]:4318
Everything is ready. Begin running and processing data.
```

**If you see `address already in use` on port 8888:** the old collector is still holding the port. Fix:
```powershell
docker stop observation_layer-otel-collector-1
docker rm observation_layer-otel-collector-1
docker compose up -d otel-collector
```

---
## Step 4 — Verify Each Service Is Up

Open these URLs in your browser **before** running any scripts. You should see:

| Service | URL | What you should see | Login |
|---|---|---|---|
| **Phoenix** | `http://localhost:6006` | Projects page, 0 traces initially | None |
| **Langfuse** | `http://localhost:3000` | Login or dashboard page | Email + password |
| **Prometheus** | `http://localhost:9095/targets` | `otel-collector` target showing **UP** in green | None |
| **Grafana** | `http://localhost:3001` | Login page → `admin` / `admin` | admin / admin |

**Grafana after login:** you should see a folder called **"LLM Observability"** with 3 dashboards:
1. LLM Service Overview
2. Token and Cost Analytics
3. Failure and SLO Diagnostics

All dashboards will show **"No data"** or **0** at this point — that's expected because no scripts have run yet.

---
## Step 5 — Initialise the Observability Pipeline (in Python)

Now we run the actual Python code. Every script and this notebook start the same way: configure OTel traces + metrics.

In [ ]:
import os, sys, time
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'))

# --- V1: Traces ---
# configure_otel() does 3 things:
#   1. Creates ONE TracerProvider with a BatchSpanProcessor
#   2. Sets it as the global tracer provider
#   3. Calls GroqInstrumentor().instrument() ONCE (auto-instruments the Groq SDK)
from observability.instrumentation import configure_otel, workflow_span, shutdown_otel
configure_otel()

# --- V2: Metrics ---
# configure_metrics() does 2 things:
#   1. Creates ONE MeterProvider with a PeriodicExportingMetricReader (2s interval)
#   2. Creates 4 counters + 1 histogram
from observability.metrics import (
    configure_metrics, flush_metrics, shutdown_metrics,
    record_request, record_tokens, record_cost, record_attempt, record_duration,
)
configure_metrics()

# Print the config so we can verify it's correct
print('='*50)
print('  OBSERVABILITY PIPELINE READY')
print('='*50)
print(f'  Service name:    {os.getenv("OTEL_SERVICE_NAME", "llm-observability-lab")}')
print(f'  Groq model:      {os.getenv("GROQ_MODEL", "allam-2-7b")}')
print(f'  Live calls:      {"YES" if os.getenv("ALLOW_LIVE_CALLS")=="1" else "NO (dry-run)"}')
print(f'  OTLP endpoint:   {os.getenv("OTEL_EXPORTER_OTLP_ENDPOINT", "http://localhost:4318")}')
print(f'  Input price/M:   ${os.getenv("PRICE_INPUT_PER_M", "0.10")}')
print(f'  Output price/M:  ${os.getenv("PRICE_OUTPUT_PER_M", "0.20")}')
print()
print('  Traces  -> Collector -> Langfuse + Phoenix + LangSmith')
print('  Metrics -> Collector -> Prometheus -> Grafana')
print()
print('  If you see warnings about GroqInstrumentor,')
print('  make sure you are using the .venv Python, not system Python.')

### What just happened behind the scenes

| Function | What it created | Where data goes |
|---|---|---|
| `configure_otel()` | `TracerProvider` + `BatchSpanProcessor` + OTLP exporter | Spans -> `localhost:4318/v1/traces` |
| `GroqInstrumentor().instrument()` | Monkey-patched `groq.Groq.chat.completions.create()` | Every future Groq call auto-emits a span |
| `configure_metrics()` | `MeterProvider` + `PeriodicExportingMetricReader` | Metrics -> `localhost:4318/v1/metrics` |
| `meter.create_counter(...)` | 4 counter instruments | `llm_requests_total`, `llm_tokens_total`, etc. |
| `meter.create_histogram(...)` | 1 histogram with buckets at 0.1-16s | `llm_request_duration_seconds` |

> **Nothing has been sent to the collector yet.** The instruments are ready but no data has been recorded. Data is sent in batches (traces: every 5s or 32 spans; metrics: every 2s).

### What just happened behind the scenes

| Function | What it created | Where data goes |
|---|---|---|
| `configure_otel()` | `TracerProvider` + `BatchSpanProcessor` + OTLP exporter | Spans -> `localhost:4318/v1/traces` |
| `workflow_span()` | Manual parent span for logical workflow | Each trace has 1 parent + N children |
| `completions_span()` | Manual child span with clean input/output JSON | `llm.input_messages` / `llm.output_messages` |
| `configure_metrics()` | `MeterProvider` + `PeriodicExportingMetricReader` | Metrics -> `localhost:4318/v1/metrics` |
| `meter.create_counter(...)` | 4 counter instruments | `llm_requests_total`, `llm_tokens_total`, etc. |
| `meter.create_histogram(...)` | 1 histogram with buckets at 0.1-16s | `llm_request_duration_seconds` |

> **Why manual spans instead of auto-instrumentation?**
> The OpenInference Groq instrumentor serializes the entire Groq SDK request
> object (including `groq.Omit` sentinel values) as the span input. This produces
> ugly output in Langfuse like `citation_options: <groq.Omit object at 0x...>`.
> Our manual `completions_span()` sets clean `llm.input_messages` and
> `llm.output_messages` as proper JSON — Langfuse renders the prompt and response
> text beautifully in its Input/Output UI.
>
> **Nothing has been sent to the collector yet.** The instruments are ready but no data has been recorded. Data is sent in batches (traces: every 5s or 32 spans; metrics: every 2s).

In [ ]:
from observability.usage import extract_usage
from observability.pricing import estimate_cost
from observability.metadata import build_attributes
from observability.instrumentation import completions_span

ALLOW_LIVE = os.getenv('ALLOW_LIVE_CALLS', '0') == '1'
MODEL = os.getenv('GROQ_MODEL', 'allam-2-7b')
PROMPT = 'List three primary colors.'  # safe, non-sensitive prompt

print(f'Making one Groq call with model={MODEL}, prompt="{PROMPT}"')
print(f'Live calls enabled: {ALLOW_LIVE}')
print()

# workflow_span creates a PARENT span.
# completions_span creates a CHILD span with clean input/output JSON.
with workflow_span(
    'demo.single_call',
    attributes=build_attributes(model=MODEL, operation='workflow'),
) as span:
    start = time.perf_counter()

    with completions_span(
        model=MODEL,
        prompt=PROMPT,
        max_tokens=64,
        temperature=0.0,
    ) as cspan:
        if ALLOW_LIVE:
            from groq import Groq
            client = Groq(api_key=os.environ['GROQ_API_KEY'])
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{'role': 'user', 'content': PROMPT}],
                max_tokens=64,
                temperature=0.0,
            )
            usage = extract_usage(resp)
            answer = resp.choices[0].message.content
            # Set clean output on the Completions span
            import json
            cspan.set_attribute('llm.output_messages',
                json.dumps([{'role': 'assistant', 'content': answer}], ensure_ascii=False))
            cspan.set_attribute('gen_ai.completion', answer)
            cspan.set_attribute('gen_ai.usage.input_tokens', usage.input_tokens)
            cspan.set_attribute('gen_ai.usage.output_tokens', usage.output_tokens)
            cspan.set_attribute('llm.token_count.prompt', usage.input_tokens)
            cspan.set_attribute('llm.token_count.completion', usage.output_tokens)
            cspan.set_attribute('llm.token_count.total', usage.total_tokens)
        else:
            # Dry-run: no API call, fake usage numbers
            usage = extract_usage({'usage': {'prompt_tokens': 12, 'completion_tokens': 14, 'total_tokens': 26}})
            answer = '(dry-run -- set ALLOW_LIVE_CALLS=1 for real calls)'

    duration = time.perf_counter() - start

    # V2: record all metrics for this call
    record_request(model=MODEL, workflow='demo.single_call', result='success')
    record_tokens(input_tokens=usage.input_tokens, output_tokens=usage.output_tokens,
                  model=MODEL, workflow='demo.single_call')
    record_duration(duration_s=duration, model=MODEL, workflow='demo.single_call', result='success')
    cost = estimate_cost(usage, MODEL)
    if cost:
        record_cost(cost_usd=cost.total_cost, model=MODEL, workflow='demo.single_call')

    # Get the trace/span IDs so we can find this trace in the UIs
    ctx = span.get_span_context()
    trace_id = f'{ctx.trace_id:032x}'
    span_id = f'{ctx.span_id:016x}'

    print(f'  Response:       {answer[:80]}')
    print(f'  Input tokens:   {usage.input_tokens}')
    print(f'  Output tokens:  {usage.output_tokens}')
    print(f'  Duration:       {duration:.3f}s')
    print(f'  Est. cost:      ~${cost.total_cost:.6f}' if cost else '  Est. cost:      missing pricing')
    print()
    print(f'  TRACE ID:       {trace_id}')
    print(f'  SPAN ID:        {span_id}')
    print()
    print('  -> Copy the TRACE ID and search for it in Phoenix / Langfuse / LangSmith')

### What to check in the UIs after Step 6

Wait 5-10 seconds for the collector to flush, then check:

**1. Phoenix** (`http://localhost:6006`)
- Go to Projects -> `default` -> Traces tab
- You should see **1 trace** named `demo.single_call`
- Click it -> see the waterfall:
  - `demo.single_call` (parent, kind=llm)
  - `Completions` (child, kind=llm) with clean input/output JSON, model, tokens

**2. Langfuse** (`http://localhost:3000`)
- Go to your project -> Tracing
- You should see **1 trace** named `demo.single_call`
- Latency column shows the duration
- Tokens column shows `12 -> 14 (sum 26)`
- Cost column shows `$0.000270` (at premium pricing: $5/M input, $15/M output)
- **To see clean input/output text:** Click the trace → click the **`Completions`**
  child observation (NOT the parent workflow span) → the Preview tab shows:
  - **Input:** "List three primary colors." (clean prompt text, no `groq.Omit` objects)
  - **Output:** "The three primary colors are red, blue, and yellow." (clean response)
  - The Metadata tab shows all 14 span attributes including `gen_ai.prompt`,
    `gen_ai.completion`, `llm.input_messages`, `llm.output_messages`, token counts

**3. Grafana** (`http://localhost:3001`)
- Open **LLM Service Overview** dashboard
- "Total Requests" stat should show **1**
- "Success Rate" should show **100%** (green)

**4. Prometheus** (`http://localhost:9095`)
- Go to the query page and run: `llm_obs_llm_requests_total`
- You should see a result with value `1` and labels showing `workflow=demo.single_call`

---
## Step 7 — Run Multiple Calls (Parent -> Children)

This runs 3 sequential calls under one workflow span. You'll see 1 parent + 3 children in the trace waterfall.

In [ ]:
PROMPTS = [
    'List three primary colors.',
    'Name a planet in our solar system.',
    'What is the chemical symbol for water?',
]

print(f'Running 3 sequential calls under one workflow span')
print()

with workflow_span(
    'demo.multi_call',
    attributes=build_attributes(model=MODEL, operation='workflow'),
) as root:
    root_start = time.perf_counter()
    call_results = []

    for i, prompt in enumerate(PROMPTS):
        call_start = time.perf_counter()

        if ALLOW_LIVE:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=64, temperature=0.0,
            )
            u = extract_usage(resp)
            ans = resp.choices[0].message.content[:40]
        else:
            u = extract_usage({'usage': {'prompt_tokens': 15+i, 'completion_tokens': 20+i*10, 'total_tokens': 35+i*11}})
            ans = '(dry-run)'

        dur = time.perf_counter() - call_start

        # Record metrics per call
        record_request(model=MODEL, workflow='demo.multi_call', result='success')
        record_tokens(input_tokens=u.input_tokens, output_tokens=u.output_tokens,
                      model=MODEL, workflow='demo.multi_call')
        record_duration(duration_s=dur, model=MODEL, workflow='demo.multi_call', result='success')
        c = estimate_cost(u, MODEL)
        if c:
            record_cost(cost_usd=c.total_cost, model=MODEL, workflow='demo.multi_call')

        call_results.append({'n': i+1, 'in': u.input_tokens, 'out': u.output_tokens, 'dur': dur, 'ans': ans})
        print(f'  call {i+1}: in={u.input_tokens:>3} out={u.output_tokens:>3} dur={dur:.3f}s  "{ans}"')

    root_dur = time.perf_counter() - root_start
    total_in = sum(r['in'] for r in call_results)
    total_out = sum(r['out'] for r in call_results)

    print()
    print('  --- workflow summary (root span) ---')
    print(f'  root duration:    {root_dur:.3f}s  (wall clock)')
    print(f'  sum(child dur):   {sum(r["dur"] for r in call_results):.3f}s')
    print(f'  total in tokens:  {total_in}')
    print(f'  total out tokens: {total_out}')
    print()
    print('  NOTE: root span has NO tokens -- only children do.')
    print('  This prevents double counting in dashboards.')

### What to check in the UIs after Step 7

**Phoenix:**
- You should now see **2 traces** total
- The new trace `demo.multi_call` -- click it and expand the waterfall:
  - `demo.multi_call` (parent)
  - `Completions` (child 1)
  - `Completions` (child 2)
  - `Completions` (child 3)
- Each child shows its own token counts and latency

**Langfuse:**
- Tracing page shows 2 traces
- The `demo.multi_call` trace shows observation count in the Levels column
- Tokens column shows the sum of all children
- **Important — where to see input/output:** Click a trace to open its detail page.
  You'll see the parent workflow span (e.g. `demo.multi_call`) and child observations
  underneath. The parent workflow span shows `Input: null` — **this is by design**.
  Manual workflow spans don't record prompt text. To see the actual LLM input/output:
  1. Expand the trace to see child observations
  2. Click on a **`Completions`** child observation (not the parent)
  3. The right panel shows the full input prompt and output response as JSON
  4. You'll also see `gen_ai.prompt`, `gen_ai.completion`, token counts, and model name

**Grafana -> LLM Service Overview:**
- "Total Requests" now shows **4** (1 from Step 6 + 3 from Step 7)
- "Requests by Workflow" chart shows 2 bars: `demo.single_call` (1) and `demo.multi_call` (3)

---

### Enable Cost Dashboard in Langfuse (one-time setup)

For the Langfuse **Cost dashboard** to show real dollar amounts instead of `$0`,
you need to configure **Model Definitions** with pricing for your model:

1. Open Langfuse → your project → **Settings** → **Model Definitions**
2. Click **"Add Model Definition"**
3. Fill in:
   - **Model Name:** `allam-2-7b`
   - **Match pattern:** `(?i)^(allam-2-7b)$`
   - **Input price:** `$0.000005` per token (= $5.00 per million tokens)
   - **Output price:** `$0.000015` per token (= $15.00 per million tokens)
   - **Tokenizer:** `None` (or `OpenAI` as a fallback)
4. Click **Submit**

After this, Langfuse automatically calculates costs for all past and future traces
that use `allam-2-7b`. The **Home** dashboard will show:
- **Model costs** card with total USD
- **Cost by model** chart
- **Usage by model** chart (input vs output tokens)
- **User consumption** chart

> **Note:** The prices above are illustrative. Replace them with your model's actual
> per-token pricing from your provider's pricing page.

---
## Step 8 — Simulate Failures (No Real API Calls)

This step uses a **fake provider** -- no Groq API is called, no quota is consumed.
We simulate timeout, 429 rate-limit, and transient-error-then-success scenarios.

Each retry attempt gets its own child span so you can see them individually in the trace UIs.

In [ ]:
from opentelemetry import trace as trace_api

tracer = trace_api.get_tracer('demo.failures')

def fake_call(scenario, attempt):
    '''Deterministic fake -- never touches the real Groq API.'''
    import time as _t; _t.sleep(0.02)  # tiny delay so latency is visible
    if scenario == 'timeout':
        raise TimeoutError('simulated read timeout')
    if scenario == 'rate_limit':
        raise ConnectionError('429 Too Many Requests')
    if scenario == 'transient':
        if attempt < 2:
            raise RuntimeError('500 Internal Server Error (transient)')
        return 'ok'

scenarios = ['timeout', 'rate_limit', 'transient']

for scenario in scenarios:
    print(f'\n--- {scenario} scenario ---')
    with workflow_span(f'demo.failure_{scenario}',
                       attributes=build_attributes(model=MODEL, operation='workflow')):
        for attempt in range(1, 4):  # max 3 attempts
            with tracer.start_as_current_span(
                f'demo.{scenario}.attempt_{attempt}',
                attributes=build_attributes(model=MODEL, operation='chat',
                                            extra={'retry.attempt': attempt, 'retry.scenario': scenario}),
            ) as span:
                try:
                    result = fake_call(scenario, attempt)
                    span.set_attribute('retry.outcome', 'success')
                    record_attempt(outcome='success', model=MODEL, workflow=f'demo.{scenario}', attempt_number=attempt)
                    record_request(model=MODEL, workflow=f'demo.{scenario}', result='success')
                    print(f'  attempt {attempt}: SUCCESS')
                    break
                except Exception as exc:
                    span.record_exception(exc)
                    span.set_status(trace_api.Status(trace_api.StatusCode.ERROR, str(exc)))
                    span.set_attribute('retry.outcome', 'error')
                    span.set_attribute('error.type', type(exc).__name__)
                    record_attempt(outcome=type(exc).__name__, model=MODEL, workflow=f'demo.{scenario}', attempt_number=attempt)
                    record_duration(duration_s=0.02, model=MODEL, workflow=f'demo.{scenario}', result=type(exc).__name__)
                    print(f'  attempt {attempt}: {type(exc).__name__}: {exc}')
                    if attempt == 3:
                        record_request(model=MODEL, workflow=f'demo.{scenario}', result=type(exc).__name__)
                        print(f'  -> exhausted 3 attempts (logical request FAILED)')

### What to check in the UIs after Step 8

**Phoenix:**
- You should see **5 traces** total (1 + 1 + 3 failure traces)
- `demo.failure_timeout` -- 3 child spans, all with status=ERROR, red indicator
- `demo.failure_rate_limit` -- same pattern, 3 error spans
- `demo.failure_transient` -- 1 error span (attempt 1) + 1 success span (attempt 2)
- Click any error span -> see the exception details in the side panel

**Langfuse:**
- Tracing page shows error icon on failure traces
- `demo.failure_timeout` shows 3 error observations
- `demo.failure_transient` shows 1 error + 1 success

**Grafana -> Failure and SLO Diagnostics:**
- "Error Count" now shows **2** (timeout + rate_limit exhausted)
- "Retry Attempts" shows failed attempts count
- "SLO: Success Rate" drops below 100% because of the 2 failures
- "Errors by Type" chart shows bars for `TimeoutError` and `ConnectionError`

---
## Step 9 — Query Prometheus Directly

Let's verify what Prometheus actually stored. This queries the Prometheus HTTP API from Python.

In [ ]:
import urllib.request, urllib.parse, json

def prom_query(query):
    '''Run a PromQL query against the local Prometheus.'''
    url = f'http://localhost:9095/api/v1/query?query={urllib.parse.quote(query)}'
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            data = json.loads(r.read())
        if data['status'] == 'success' and data['data']['result']:
            return data['data']['result']
    except Exception as e:
        return [{'error': str(e)}]
    return []

# Flush metrics first, then wait for Prometheus to scrape
flush_metrics()
time.sleep(6)

queries = [
    ('Total Requests',        'sum(llm_obs_llm_requests_total)'),
    ('Successful Requests',   'sum(llm_obs_llm_requests_total{result="success"})'),
    ('Failed Requests',       'sum(llm_obs_llm_requests_total{result!="success"})'),
    ('Total Tokens',          'sum(llm_obs_llm_tokens_total)'),
    ('Input Tokens',          'sum(llm_obs_llm_tokens_total{token_type="input"})'),
    ('Output Tokens',         'sum(llm_obs_llm_tokens_total{token_type="output"})'),
    ('Total API Attempts',    'sum(llm_obs_llm_api_attempts_total)'),
    ('Failed Attempts',       'sum(llm_obs_llm_api_attempts_total{outcome!="success"})'),
    ('P95 Latency (s)',       'histogram_quantile(0.95, sum by (le) (llm_obs_llm_request_duration_seconds_bucket))'),
    ('Success Rate',          'sum(llm_obs_llm_requests_total{result="success"}) / clamp_min(sum(llm_obs_llm_requests_total), 1)'),
]

print(f'{"Metric":<25} {"Value":>15}')
print('-' * 42)
for name, query in queries:
    result = prom_query(query)
    if result and 'error' not in result[0]:
        val = result[0]['value'][1]
        try:
            v = float(val)
            if 'Rate' in name:
                print(f'{name:<25} {v:>14.1%}')
            elif 'P95' in name:
                print(f'{name:<25} {v:>14.3f}s')
            elif 'Tokens' in name:
                print(f'{name:<25} {v:>15.0f}')
            else:
                print(f'{name:<25} {v:>15.0f}')
        except ValueError:
            print(f'{name:<25} {val:>15}')
    else:
        err = result[0].get('error', 'no data') if result else 'no data'
        print(f'{name:<25} {err:>15}')

### Understanding the Prometheus output

| Metric | What it tells you |
|---|---|
| Total Requests | How many logical requests were recorded (includes successes + failures) |
| Successful vs Failed | Breakdown by `result` label -- `success` vs error type names |
| Total Tokens | Sum of input + output tokens across all calls |
| Input vs Output | Split by `token_type` label -- helps see if output dominates cost |
| Total API Attempts | Every individual API call (including retries) -- always >= Total Requests |
| Failed Attempts | Attempts that errored -- shows retry activity |
| P95 Latency | 95th percentile from the histogram -- if > 4s, latency SLO is violated |
| Success Rate | Successful / Total -- if < 99.5%, success SLO is violated |

---
## Step 10 — Evaluate SLOs

We compute the two internal SLOs from the Prometheus data.

- **SLO 1:** 7-day success >= 99.5%
- **SLO 2:** >= 95% under 4 seconds

If total samples < 100, we show an **insufficient-data** state instead of a misleading percentage.

In [ ]:
from observability.slos import (
    evaluate_success_slo, evaluate_latency_slo, format_slo_report, MIN_SAMPLES
)

# Pull current values from Prometheus
total_r = prom_query('sum(llm_obs_llm_requests_total)')
succ_r = prom_query('sum(llm_obs_llm_requests_total{result="success"})')
under4_r = prom_query('sum(llm_obs_llm_request_duration_seconds_bucket{le="4"})')

total = int(float(total_r[0]['value'][1])) if total_r and 'error' not in total_r[0] else 0
success = int(float(succ_r[0]['value'][1])) if succ_r and 'error' not in succ_r[0] else 0
under4 = int(float(under4_r[0]['value'][1])) if under4_r and 'error' not in under4_r[0] else 0

print(f'Current sample count: {total}')
print(f'Minimum for SLO eval: {MIN_SAMPLES}')
print(f'Successful:           {success}')
print(f'Under 4s:             {under4}')
print()

success_slo = evaluate_success_slo(total, success)
latency_slo = evaluate_latency_slo(success, under4)
print(format_slo_report(success_slo, latency_slo))

### Reading the SLO report

**If you see "INSUFFICIENT DATA":**
- You have fewer than 100 samples. Run the synthetic demo (next step) to generate 500 samples quickly.

**If you see "MET":**
- The SLO target is met. Error budget is not exhausted.

**If you see "VIOLATED":**
- The SLO target is missed. Error budget consumed > 100%.
- With the failures from Step 8, the success rate SLO may be violated depending on total calls.

**The hypothetical 99% monthly SLA** at the bottom is documentation-only -- it is NOT computed and NOT a provider's contractual SLA.

---
## Step 11 — Populate Dashboards with Synthetic Data (Offline)

If you want realistic-looking dashboards without making hundreds of live API calls, run the synthetic demo. It generates 500 observations with ~95% success rate, realistic latency, and token distributions -- **without touching the Groq API**.

**Run in a terminal:**
```powershell
cd d:\Observation_Layer
$env:PYTHONPATH = "d:\Observation_Layer"
$env:PYTHONIOENCODING = "utf-8"
.venv\Scripts\python.exe examples\06_synthetic_metrics.py
```

**Expected output:**
```
============================================================
  SYNTHETIC -- no live API calls
  Generating 500 synthetic metric observations...
============================================================
  ... 100/500 observations emitted
  ... 200/500 observations emitted
  ... 300/500 observations emitted
  ... 400/500 observations emitted
  ... 500/500 observations emitted

  Total observations:   500
  Successful:           ~469
  Errors:               ~31
  Total est. cost:      ~$0.017
```

**After running, check Grafana:**

| Dashboard | Panel | What you'll see |
|---|---|---|
| LLM Service Overview | Total Requests | ~504 (4 from notebook + 500 synthetic) |
| LLM Service Overview | Success Rate | ~93% (yellow -- below 99.5% SLO) |
| LLM Service Overview | P95 Latency | ~0.9s (green -- under 4s SLO) |
| Token and Cost Analytics | Total Tokens | ~185,000 |
| Token and Cost Analytics | Estimated Cost | ~$0.034 |
| Failure and SLO Diagnostics | Error Count | ~33 |
| Failure and SLO Diagnostics | Error Budget Consumed | >100% (red -- exhausted) |
| Failure and SLO Diagnostics | Total Samples | ~504 (green -- above 100 minimum) |

---
## Step 12 — Run the Example Scripts from the Terminal

The `examples/` folder has 6 scripts. Run them from a terminal:

```powershell
cd d:\Observation_Layer
$env:PYTHONPATH = "d:\Observation_Layer"
$env:PYTHONIOENCODING = "utf-8"
```

| Script | Command | What it does | Live calls? |
|---|---|---|---|
| 01 | `.venv\Scripts\python.exe examples\01_single_call.py` | One Groq call, verify all 3 trace platforms | Yes (if ALLOW_LIVE_CALLS=1) |
| 02 | `.venv\Scripts\python.exe examples\02_multiple_calls.py` | 3 sequential calls under one workflow span | Yes |
| 03 | `.venv\Scripts\python.exe examples\03_concurrent_calls.py` | 5 concurrent vs 5 sequential, compares p50/p95 | Yes |
| 04 | `.venv\Scripts\python.exe examples\04_failures.py` | Timeout, 429, transient errors with retries | No (fake provider) |
| 05 | `.venv\Scripts\python.exe examples\05_batch_dashboard.py` | 30 calls (15 seq + 15 conc) for dashboard data | Yes |
| 06 | `.venv\Scripts\python.exe examples\06_synthetic_metrics.py` | 500 synthetic observations for Grafana | No (offline) |

**Expected output from script 03 (concurrent):**
```
=== sequential ===
wall time:     3.381s
p50 latency:   0.441s
p95 latency:   0.451s
throughput:    1.48 calls/s

=== concurrent ===
wall time:     0.656s
p50 latency:   0.622s
p95 latency:   0.624s
throughput:    7.62 calls/s

speedup (seq/conc wall): 5.16x
```

> **Why concurrent p50 is higher:** individual calls slow down when 5 run in parallel (they compete for the same API rate budget), but total throughput is 5x higher. This is exactly the trade-off dashboards make visible.

---
## Step 13 — Flush and Shutdown

**This is critical.** Short-running Python scripts buffer spans and metrics in memory. If you don't flush before the process exits, data may never reach the collector.

All example scripts call `shutdown_all()` in a `finally` block. This notebook must do the same.

In [ ]:
# Force-flush all pending telemetry before exiting
flush_metrics()      # export any buffered metrics right now
shutdown_metrics()   # shut down the meter provider
shutdown_otel()      # flush + shut down the tracer provider

print('='*50)
print('  ALL TELEMETRY FLUSHED AND SHUT DOWN')
print('='*50)
print()
print('  What was flushed:')
print('    - Traces  -> localhost:4318/v1/traces  -> collector -> 3 backends')
print('    - Metrics -> localhost:4318/v1/metrics -> collector -> Prometheus')
print()
print('  Wait 5-10 seconds, then check your dashboards:')
print('    Phoenix:    http://localhost:6006')
print('    Langfuse:   http://localhost:3000')
print('    Grafana:    http://localhost:3001')
print('    Prometheus: http://localhost:9095')

---
## Quick Reference Card

### Terminal commands cheat sheet

```powershell
# --- Setup (first time) ---
cd d:\Observation_Layer
cp .env.example .env                    # then edit .env with your 4 keys
.\infrastructure\sync-keys.ps1          # auto-generate collector.env from .env + restart collector
.venv\Scripts\Activate.ps1              # activate venv
pip install -e ".[dev]"                 # install dependencies

# --- Start infrastructure (every time) ---
cd d:\langfuse; docker compose up -d    # start Langfuse first
cd d:\Observation_Layer; docker compose up -d  # then start this stack
docker logs observation_layer-otel-collector-1 --tail 3   # verify collector

# --- If you changed Langfuse keys in .env ---
.\infrastructure\sync-keys.ps1          # re-syncs collector.env + restarts collector

# --- Run scripts ---
$env:PYTHONPATH = "d:\Observation_Layer"
$env:PYTHONIOENCODING = "utf-8"
.venv\Scripts\python.exe examples\01_single_call.py
.venv\Scripts\python.exe examples\06_synthetic_metrics.py   # offline, no API

# --- Tests ---
.venv\Scripts\python.exe -m pytest -v

# --- Stop everything ---
docker compose down
```

### URLs

| Service | URL | Login |
|---|---|---|
| Phoenix (traces) | `http://localhost:6006` | none |
| Langfuse (traces) | `http://localhost:3000` | your email + password |
| LangSmith (SaaS traces) | `https://smith.langchain.com` | your LangSmith account |
| Prometheus (metrics) | `http://localhost:9095` | none |
| Grafana (dashboards) | `http://localhost:3001` | admin / admin |

### Grafana dashboards (3 provisioned automatically)

| # | Dashboard | Key panels |
|---|---|---|
| 1 | LLM Service Overview | Total Requests, Success Rate, P95 Latency, Error Budget, Latency P50/P95/P99 |
| 2 | Token and Cost Analytics | Total/Input/Output Tokens, Estimated Cost, Tokens by Model, Cost by Model |
| 3 | Failure and SLO Diagnostics | Error Count, Retry Attempts, SLO Success Rate, SLO % Under 4s, Errors by Type, Error Budget Consumed |

### Env keys you need (and what breaks without them)

| Key | Required for | Without it |
|---|---|---|
| `GROQ_API_KEY` | Live API calls | Scripts run in dry-run mode |
| `LANGFUSE_PUBLIC_KEY` + `SECRET_KEY` | Langfuse trace export | Traces won't appear in Langfuse |
| `LANGSMITH_API_KEY` | LangSmith trace export | Traces won't appear in LangSmith |
| `ALLOW_LIVE_CALLS=1` | Real Groq calls | All scripts emit synthetic spans only |

### Files that hold secrets (never commit)

| File | What's in it | How to update |
|---|---|---|
| `.env` | GROQ_API_KEY, LANGFUSE keys, LANGSMITH key, pricing | Edit manually |
| `infrastructure/collector.env` | LANGFUSE_AUTH_HEADER, LANGSMITH_API_KEY | Auto-generated by `sync-keys.ps1` — do NOT edit manually |

### Concurrent calls — what changed

When running concurrent calls (script 03 or 05), each call now gets its **own individual child span**
under the workflow parent span. This was a bug where OTel context didn't propagate to thread executors.

**Before the fix:** all concurrent calls shared the parent's context or had no span at all —
the trace waterfall showed only 1 span instead of N+1.

**After the fix:** each concurrent call attaches the captured OTel context, producing:
```
concurrent.workflow              (parent)
  ├── Completions  (call 0)     (child — own span ID)
  ├── Completions  (call 1)     (child — own span ID)
  ├── Completions  (call 2)     (child — own span ID)
  ├── Completions  (call 3)     (child — own span ID)
  └── Completions  (call 4)     (child — own span ID)
```

This works via `opentelemetry.context.get_current()` + `attach()` / `detach()` in each thread.